# Лекција 13 - Меморија агента са Cognee знaњским графовима


## Setup

This notebook demonstrates how to build an intelligent **coding assistant** with persistent memory using [**Cognee**](https://www.cognee.ai/) knowledge graphs and the **Microsoft Agent Framework** (MAF).

Cognee transforms unstructured text into a structured, queryable knowledge graph backed by vector embeddings — giving your agent a rich, relationship-aware long-term memory.

### What You'll Learn
1. **Build Knowledge Graphs**: Transform developer profiles and best practices into structured, queryable knowledge.
2. **Integrate Cognee with MAF**: Use `@tool` functions to let an MAF agent query Cognee's knowledge graph.
3. **Session-Aware Conversations**: Maintain context across multiple questions in the same session.
4. **Long-Term Memory**: Persist important knowledge across sessions and retrieve it in new conversations.

### Prerequisites
- Python 3.9+
- Redis running locally (`docker run -d -p 6379:6379 redis`) for session management
- An LLM API key (e.g. OpenAI) — set `LLM_API_KEY` in `.env`
- `CACHING=true` in `.env` (required for Cognee sessions)
- A Microsoft Foundry project with a deployed chat model
- `AZURE_AI_PROJECT_ENDPOINT` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Типови меморије агента

Ова свеска истражује иста три типа меморије као и главна свеска из Лекције 13, али користи Cognee као бекенд за дугорочну меморију:

| Тип меморије | Механизам | Век трајања |
|---|---|---|
| **Радна** | `agent.create_session()` (MAF) | Један разговор |
| **Краткорочна** | Cognee кеш сесије (Redis) | Једна сесија |
| **Дугорочна** | Cognee граф знања + вектори | Трајна |

### Архитектура меморије Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Припремите Cognee меморију


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Део 1 — Изградња базе знања

Прикупљамо три врсте података да бисмо створили свеобухватну базу знања за нашег асистента за програмирање:

1. **Профил програмера** — лична стручност и техничка позадина
2. **Најбоље праксе у Пајтону** — Зен Пајтона са практичним смерницама
3. **Историјски разговори** — претходне сесије питања и одговора између програмера и АИ асистената


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Визуализујте граф знања

Cognee може приказати интерактивну HTML визуализацију ентитета и веза које је извукао.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Обогати меморију са Memify

`memify()` анализира граф знања и генерише интелигентна правила — идентификујући шаблоне, најбоље праксе и односе између концепата.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Део 2 — MAF агент са Cognee алаткама

Сада креирамо MAF агента који може да упитује граф знања Cognee преко `@tool` функција. Ово агенту омогућава да користи пун потенцијал семантичке претраге свестране о графу, док одржава контекст разговора кроз сесије.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Радна меморија са сесијама

`AgentSession` (креирано преко `agent.create_session()`) обезбеђује радну меморију унутар сесије. агент може да се позивa на претходне поруке, а такође и да упита дугорочни граф знања Cognee-а.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Нова сесија — дугорочно памћење остаје

Покретање нове сесије брише радну меморију, али граф знања Cognee је и даље доступан. Агенат може да преузме иста дугорочна знања у потпуно новом разговору.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резиме

У овој свесци сте направили асистента за програмирање који комбинује **МАФ радну меморију** (`agent.create_session()`) са **Когнијином дугорочном граф базом знања**.

### Шта сте научили
1. **Конструкција графа знања**: Когниј прима неструктурисани текст и гради граф + векторску меморију.
2. **Обогаћивање графа са memify**: Изведени подаци и богатији односи на врху постојећег графа.
3. **Интеграција МАФ + Когниј**: `@tool` функције омогућавају МАФ агенту природно испитивање Когнијиног графа.
4. **Радна меморија + дугорочна меморија**: `AgentSession` (путем `agent.create_session()`) обезбеђује контекст сесије док Когниј пружа константно знање.
5. **Филтрирана претрага са NodeSets**: Циљане подсетове графа знања (нпр. само принципе).

### Главне поуке
- **Когниј** претвара необрађени текст у структурисану меморију с односима — моћнију од обичне векторске складиште.
- **`@tool` функције** чисто повезују МАФ агенте и спољне системе знања.
- **`AgentSession`** (путем `agent.create_session()`) чува контекст по разговору засебно од дуготрајног знања.
- Исти граф знања служи више сесија и агената.

### Примена у стварном свету
- **Развојни копилоти**: Преглед кода, анализа инцидената, асистенти за архитектуру
- **Копилоти за кориснике**: Агенти за подршку преко документације, ЧПП и CRM записа
- **Унутрашњи експертски копилоти**: Политика, правни или безбедносни асистенти који разматрају смернице
- **Уједињени слојеви података**: Комбинују структуриране и неструктуриране податке у један упитни граф

### Следећи кораци
- Експериментишите с временском свешћу у Когнију
- Дефинишите OWL онтологију за квалитет графа специфичан за домен
- Додајте повратне информације корисника за побољшање проналаска током времена
- Масовно примените на мулти-агентске системе који деле исти Когниј слој меморије


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
